# HOLa-style predicate description low-rank decomposition

This notebook encodes one text description per scene-graph predicate with the repository-local CLIP text encoder, then follows the HOLa paper's learnable feature decomposition objective:

$$F \approx W B^\top$$

- `F`: CLIP text features, shape `[num_predicates, clip_dim]`
- `W`: learnable predicate-specific weights, shape `[num_predicates, rank]`
- `B`: learnable shared basis features, shape `[clip_dim, rank]`
- `rank`: low-rank dimension `m`

The optimization uses reconstruction loss, L1 sparsity on `W`, and an orthogonality constraint on `B`. If a predicate has no LLM description yet, it falls back to `a photo of {predicate}` and records that in metadata.


## 1. Configuration

Edit this cell before running. For a quick smoke test without the VG dict, set `PREDICATE_JSON = None` and use `PREDICATE_LIST`.

In [ ]:
import os

REPO_ROOT = "/Users/shangfei/Developer/SDSGG"

# Use either PREDICATE_JSON or PREDICATE_LIST.
PREDICATE_JSON = os.path.join(REPO_ROOT, "datasets/vg/VG-SGG-dicts-with-attri.json")
PREDICATE_LIST = None  # Example: ["on", "under", "riding", "holding"]

# Optional LLM descriptions. Supported formats:
# JSON: {"on": "Objects are in contact...", "riding": "..."}
# CSV: columns must be predicate,description
DESCRIPTIONS = None

CLIP_MODEL = "ViT-B/32"  # Or a local checkpoint path, e.g. "/path/to/ViT-B-32.pt"
DEVICE = "cuda"  # Change to "cpu" if needed.
RANK = 16
BATCH_SIZE = 64
NORMALIZE_FEATURES = True
KEEP_BACKGROUND = False
PROMPT_TEMPLATE = "a photo of {predicate}"

# HOLa-style learnable factorization settings.
FACTORIZATION_STEPS = 3000
FACTORIZATION_LR = 1e-2
LAMBDA_SPARSE = 1e-3
LAMBDA_ORTHO = 1e-2
PRINT_EVERY = 250
SEED = 42

OUTPUT = os.path.join(REPO_ROOT, "outputs/hola_predicate_factors.pt")

# Optional VG triplet-statistics inputs for train/test analysis.
VG_ROIDB_FILE = os.path.join(REPO_ROOT, "datasets/vg/VG-SGG-with-attri.h5")
VG_DICT_FILE = os.path.join(REPO_ROOT, "datasets/vg/VG-SGG-dicts-with-attri.json")
ENABLE_VG_TRIPLET_STATS = True
TOPK_TRIPLETS_PER_PREDICATE = 10

# If your environment is missing CLIP tokenizer dependencies, run this once:
# %pip install ftfy regex tqdm


## 2. Imports and helper functions

In [ ]:
import csv
import json
import h5py
import sys
from typing import Dict, List, Optional, Sequence, Tuple

import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F

LOCAL_CLIP_PARENT = os.path.join(
    REPO_ROOT, "maskrcnn_benchmark", "modeling", "roi_heads", "relation_head"
)
if LOCAL_CLIP_PARENT not in sys.path:
    sys.path.insert(0, LOCAL_CLIP_PARENT)

print("Repo root:", REPO_ROOT)
print("Local CLIP parent:", LOCAL_CLIP_PARENT)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
def normalize_key(text: str) -> str:
    return " ".join(str(text).strip().lower().split())


def load_predicates_from_json(path: str) -> List[Tuple[int, str]]:
    with open(path, "r", encoding="utf-8") as handle:
        payload = json.load(handle)

    if "idx_to_predicate" in payload:
        idx_to_predicate = payload["idx_to_predicate"]
        if isinstance(idx_to_predicate, dict):
            return sorted(
                ((int(index), name) for index, name in idx_to_predicate.items()),
                key=lambda item: item[0],
            )
        return [(index, name) for index, name in enumerate(idx_to_predicate)]

    if "predicate_to_idx" in payload:
        predicate_to_idx = payload["predicate_to_idx"]
        return sorted(
            ((int(index), name) for name, index in predicate_to_idx.items()),
            key=lambda item: item[0],
        )

    raise ValueError(
        f"No predicate mapping found in {path}. Expected idx_to_predicate or predicate_to_idx."
    )


def load_predicates_from_list(predicate_list: Sequence[str]) -> List[Tuple[int, str]]:
    predicates = [item.strip() for item in predicate_list if item and item.strip()]
    if not predicates:
        raise ValueError("PREDICATE_LIST is empty after parsing.")
    return list(enumerate(predicates))


def filter_background(predicates: Sequence[Tuple[int, str]], keep_background: bool) -> List[Tuple[int, str]]:
    if keep_background:
        return list(predicates)
    return [
        (index, name)
        for index, name in predicates
        if normalize_key(name) != "__background__"
    ]

In [ ]:
def load_description_map(path: str) -> Dict[str, str]:
    ext = os.path.splitext(path)[1].lower()

    if ext == ".json":
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
        if not isinstance(payload, dict):
            raise ValueError("Description JSON must map predicate -> description.")
        return {normalize_key(key): str(value).strip() for key, value in payload.items()}

    if ext == ".csv":
        description_map = {}
        with open(path, "r", encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            required_fields = {"predicate", "description"}
            if not reader.fieldnames or not required_fields.issubset(reader.fieldnames):
                raise ValueError("Description CSV must contain predicate and description columns.")
            for row in reader:
                predicate = normalize_key(row["predicate"])
                description = row["description"].strip()
                if predicate:
                    description_map[predicate] = description
        return description_map

    raise ValueError(f"Unsupported description format: {path}. Use JSON or CSV.")


def resolve_descriptions(
    predicates: Sequence[Tuple[int, str]],
    description_map: Optional[Dict[str, str]],
    prompt_template: str,
) -> Tuple[List[str], List[bool]]:
    descriptions = []
    fallback_mask = []

    for _, predicate_name in predicates:
        description = ""
        if description_map is not None:
            description = description_map.get(normalize_key(predicate_name), "").strip()

        used_fallback = not description
        if used_fallback:
            description = prompt_template.format(predicate=predicate_name)

        descriptions.append(description)
        fallback_mask.append(used_fallback)

    return descriptions, fallback_mask

In [ ]:
def encode_text_features(
    texts: Sequence[str],
    clip_model_name: str,
    device: str,
    batch_size: int,
) -> torch.Tensor:
    try:
        from CLIP import clip
    except ModuleNotFoundError as error:
        raise ModuleNotFoundError(
            "Failed to import local CLIP dependencies. "
            "Install the missing package in this notebook kernel, e.g. `%pip install ftfy regex tqdm`. "
            f"Missing package: {error.name}"
        ) from error

    try:
        model, _ = clip.load(clip_model_name, device=device)
    except RuntimeError as error:
        raise RuntimeError(
            "Failed to load CLIP model. If weights are not cached locally, pass a local checkpoint "
            "path in CLIP_MODEL or prepare ~/.cache/clip first. "
            f"Original error: {error}"
        ) from error

    model.eval()
    features = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = list(texts[start:start + batch_size])
            tokens = clip.tokenize(batch_texts).to(device)
            batch_features = model.encode_text(tokens).float().cpu()
            features.append(batch_features)

    return torch.cat(features, dim=0)


def effective_rank(requested_rank: int, features: torch.Tensor) -> int:
    if requested_rank < 1:
        raise ValueError("RANK must be positive.")
    return min(requested_rank, features.shape[0], features.shape[1])


def orthogonality_loss(B: torch.Tensor) -> torch.Tensor:
    """Penalize non-orthogonal basis columns, matching HOLa's shared basis constraint."""
    gram = B.t() @ B
    identity = torch.eye(gram.shape[0], device=gram.device, dtype=gram.dtype)
    off_diagonal = gram - identity
    return off_diagonal.pow(2).mean()


def compute_low_rank_factors_learnable(
    features: torch.Tensor,
    requested_rank: int,
    steps: int,
    lr: float,
    lambda_sparse: float,
    lambda_ortho: float,
    seed: int,
    print_every: int,
):
    """Optimize F ~= W @ B.T with HOLa-style sparse W and orthogonal B losses."""
    if steps < 1:
        raise ValueError("FACTORIZATION_STEPS must be positive.")

    target = features.float()
    rank = effective_rank(requested_rank, target)
    num_predicates, feature_dim = target.shape

    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)

    W = torch.randn(num_predicates, rank, generator=generator, dtype=target.dtype) * 0.02
    B = torch.randn(feature_dim, rank, generator=generator, dtype=target.dtype) * 0.02

    # Start B near an orthonormal basis for a stable HOLa-style orthogonal constraint.
    with torch.no_grad():
        q, _ = torch.linalg.qr(torch.randn(feature_dim, rank, generator=generator, dtype=target.dtype), mode="reduced")
        B.copy_(q)

    W = torch.nn.Parameter(W)
    B = torch.nn.Parameter(B)
    optimizer = torch.optim.Adam([W, B], lr=lr)
    history = []

    for step in range(1, steps + 1):
        reconstructed = W @ B.t()
        recon_loss = F.mse_loss(reconstructed, target)
        sparse_loss = W.abs().sum(dim=1).mean()
        ortho_loss = orthogonality_loss(B)
        total_loss = recon_loss + lambda_sparse * sparse_loss + lambda_ortho * ortho_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        if print_every and (step == 1 or step % print_every == 0 or step == steps):
            record = {
                "step": step,
                "total_loss": float(total_loss.detach().cpu()),
                "recon_loss": float(recon_loss.detach().cpu()),
                "sparse_loss": float(sparse_loss.detach().cpu()),
                "ortho_loss": float(ortho_loss.detach().cpu()),
            }
            history.append(record)
            print(record)

    with torch.no_grad():
        final_W = W.detach().cpu()
        final_B = B.detach().cpu()
        reconstructed = final_W @ final_B.t()
        reconstruction_error = torch.norm(target.cpu() - reconstructed, p="fro")
        final_recon_loss = F.mse_loss(reconstructed, target.cpu())
        final_sparse_loss = final_W.abs().sum(dim=1).mean()
        final_ortho_loss = orthogonality_loss(final_B)
        final_total_loss = final_recon_loss + lambda_sparse * final_sparse_loss + lambda_ortho * final_ortho_loss

    return {
        "W": final_W,
        "B": final_B,
        "M": final_B.t(),  # Compatibility alias: F ~= W @ M, while HOLa notation uses B.T.
        "reconstructed": reconstructed,
        "rank": rank,
        "reconstruction_error": reconstruction_error,
        "final_total_loss": final_total_loss,
        "final_recon_loss": final_recon_loss,
        "final_sparse_loss": final_sparse_loss,
        "final_ortho_loss": final_ortho_loss,
        "loss_history": history,
    }


## 3. Load predicates and descriptions

In [ ]:
if PREDICATE_JSON and PREDICATE_LIST:
    raise ValueError("Use only one of PREDICATE_JSON or PREDICATE_LIST.")
if not PREDICATE_JSON and not PREDICATE_LIST:
    raise ValueError("Set either PREDICATE_JSON or PREDICATE_LIST.")

if PREDICATE_JSON:
    predicates = load_predicates_from_json(PREDICATE_JSON)
else:
    predicates = load_predicates_from_list(PREDICATE_LIST)

predicates = filter_background(predicates, KEEP_BACKGROUND)
if not predicates:
    raise ValueError("No predicates remain after background filtering.")

description_map = load_description_map(DESCRIPTIONS) if DESCRIPTIONS else None
descriptions, fallback_mask = resolve_descriptions(predicates, description_map, PROMPT_TEMPLATE)

predicate_indices = [index for index, _ in predicates]
predicate_names = [name for _, name in predicates]

print(f"Loaded predicates: {len(predicate_names)}")
print(f"Fallback descriptions: {sum(fallback_mask)}")
print("First predicates:", predicate_names[:10])
print("First descriptions:", descriptions[:3])

## 4. Encode text with CLIP

In [ ]:
print("Step 4: Encode predicate descriptions with CLIP.")
print("- Input text count:", len(descriptions))
print("- CLIP model:", CLIP_MODEL)
print("- Device:", DEVICE)
print("- Normalize features:", NORMALIZE_FEATURES)

features = encode_text_features(
    texts=descriptions,
    clip_model_name=CLIP_MODEL,
    device=DEVICE,
    batch_size=BATCH_SIZE,
)

if NORMALIZE_FEATURES:
    features = F.normalize(features, dim=-1)

print("Done encoding text features.")
print("features shape:", tuple(features.shape), "dtype:", features.dtype)
print("feature norm range:", float(features.norm(dim=-1).min()), float(features.norm(dim=-1).max()))
print("Interpretation: each row is one predicate description embedding; rows are later approximated by W @ B.T.")


## 5. Compute HOLa-style learnable low-rank factors


In [ ]:
print("Step 5: Optimize learnable HOLa-style factors W and B.")
print("Objective: minimize recon_loss + lambda_sparse * ||W||_1 + lambda_ortho * orthogonality(B)")
print("Target matrix F shape:", tuple(features.shape))
print("Requested rank m:", RANK)
print("Optimization steps:", FACTORIZATION_STEPS)
print("Learning rate:", FACTORIZATION_LR)
print("lambda_sparse:", LAMBDA_SPARSE)
print("lambda_ortho:", LAMBDA_ORTHO)

factor_payload = compute_low_rank_factors_learnable(
    features=features,
    requested_rank=RANK,
    steps=FACTORIZATION_STEPS,
    lr=FACTORIZATION_LR,
    lambda_sparse=LAMBDA_SPARSE,
    lambda_ortho=LAMBDA_ORTHO,
    seed=SEED,
    print_every=PRINT_EVERY,
)

W = factor_payload["W"]
B = factor_payload["B"]
M = factor_payload["M"]
reconstructed = factor_payload["reconstructed"]
rank = factor_payload["rank"]

print("\nOptimization finished.")
print("Requested rank:", RANK)
print("Effective rank:", rank)
print("W shape [num_predicates, rank]:", tuple(W.shape))
print("B shape [clip_dim, rank]:", tuple(B.shape))
print("M = B.T shape [rank, clip_dim]:", tuple(M.shape))
print("reconstructed shape:", tuple(reconstructed.shape))
print("reconstruction_error Frobenius:", float(factor_payload["reconstruction_error"]))
print("final_total_loss:", float(factor_payload["final_total_loss"]))
print("final_recon_loss:", float(factor_payload["final_recon_loss"]))
print("final_sparse_loss:", float(factor_payload["final_sparse_loss"]))
print("final_ortho_loss:", float(factor_payload["final_ortho_loss"]))
print("Interpretation: W is predicate-specific; B is the shared semantic basis across predicates.")


## 6. Diagnostics A: does the low-rank factorization itself look reasonable?

This section validates the decomposition before using it for any downstream SGG model.

We check four things:

1. **Reconstruction quality**: whether `W @ B.T` preserves CLIP text features.
2. **Loss curve**: whether optimization converged instead of still drifting.
3. **Orthogonality of `B`**: whether basis columns are disentangled rather than duplicated.
4. **Sparsity of `W`**: whether each predicate uses only a small subset of basis features.


In [ ]:
print("Diagnostic A1: Reconstruction quality")

residual = features.cpu() - reconstructed
per_predicate_error = residual.norm(dim=1)
feature_norm = features.cpu().norm(dim=1).clamp_min(1e-12)
relative_error = per_predicate_error / feature_norm
cosine_to_reconstruction = F.cosine_similarity(features.cpu(), reconstructed, dim=1)

print("What this means:")
print("- Low relative error means W @ B.T keeps most of the original CLIP description feature.")
print("- High cosine means the reconstructed feature points in a similar semantic direction.")
print("\nGlobal reconstruction stats:")
print("Frobenius error:", float(factor_payload["reconstruction_error"]))
print("Mean per-predicate L2 error:", float(per_predicate_error.mean()))
print("Median per-predicate L2 error:", float(per_predicate_error.median()))
print("Mean relative error:", float(relative_error.mean()))
print("Median relative error:", float(relative_error.median()))
print("Mean cosine(F, reconstructed):", float(cosine_to_reconstruction.mean()))
print("Min cosine(F, reconstructed):", float(cosine_to_reconstruction.min()))

worst_k = min(10, len(predicate_names))
worst_indices = torch.argsort(relative_error, descending=True)[:worst_k].tolist()
print(f"\nWorst {worst_k} reconstructed predicates by relative error:")
for row_id in worst_indices:
    print({
        "predicate_index": predicate_indices[row_id],
        "predicate": predicate_names[row_id],
        "relative_error": round(float(relative_error[row_id]), 4),
        "cosine": round(float(cosine_to_reconstruction[row_id]), 4),
        "used_fallback": bool(fallback_mask[row_id]),
    })

plt.figure(figsize=(6, 4))
plt.hist(relative_error.numpy(), bins=30)
plt.title("Per-predicate relative reconstruction error")
plt.xlabel("||F_i - W_i B.T|| / ||F_i||")
plt.ylabel("# predicates")
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(cosine_to_reconstruction.numpy(), bins=30)
plt.title("Cosine similarity: original F vs reconstructed")
plt.xlabel("cosine(F_i, reconstructed_i)")
plt.ylabel("# predicates")
plt.show()


In [ ]:
print("Diagnostic A2: Optimization loss curve")

history = factor_payload["loss_history"]
if not history:
    print("No loss history was recorded. Set PRINT_EVERY to a positive value.")
else:
    steps = [item["step"] for item in history]
    total_losses = [item["total_loss"] for item in history]
    recon_losses = [item["recon_loss"] for item in history]
    sparse_losses = [item["sparse_loss"] for item in history]
    ortho_losses = [item["ortho_loss"] for item in history]

    print("First record:", history[0])
    print("Last record:", history[-1])
    print("What to look for: total_loss and recon_loss should decrease and then stabilize.")

    plt.figure(figsize=(7, 4))
    plt.plot(steps, total_losses, label="total")
    plt.plot(steps, recon_losses, label="recon")
    plt.xlabel("optimization step")
    plt.ylabel("loss")
    plt.title("HOLa factorization losses")
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 4))
    plt.plot(steps, sparse_losses, label="sparse ||W||1")
    plt.plot(steps, ortho_losses, label="orthogonality(B)")
    plt.xlabel("optimization step")
    plt.ylabel("loss component")
    plt.title("Regularization losses")
    plt.legend()
    plt.show()


In [ ]:
print("Diagnostic A3: Basis orthogonality")

B_gram = B.t() @ B
identity = torch.eye(B_gram.shape[0])
off_diag = B_gram - torch.diag(torch.diag(B_gram))
basis_norms = B.norm(dim=0)

print("What this means:")
print("- B.T @ B should be close to identity if basis features are non-redundant.")
print("- Off-diagonal values near 0 mean different basis vectors are less correlated.")
print("\nBasis norm stats:")
print("min norm:", float(basis_norms.min()))
print("mean norm:", float(basis_norms.mean()))
print("max norm:", float(basis_norms.max()))
print("\nOrthogonality stats:")
print("mean abs off-diagonal:", float(off_diag.abs().mean()))
print("max abs off-diagonal:", float(off_diag.abs().max()))
print("mean squared B.TB-I:", float((B_gram - identity).pow(2).mean()))

plt.figure(figsize=(5, 4))
plt.imshow(B_gram.numpy(), cmap="coolwarm")
plt.colorbar()
plt.title("B.T @ B: basis correlation matrix")
plt.xlabel("basis id")
plt.ylabel("basis id")
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(off_diag.flatten().numpy(), bins=40)
plt.title("Off-diagonal values in B.T @ B")
plt.xlabel("basis correlation")
plt.ylabel("count")
plt.show()


In [ ]:
print("Diagnostic A4: Weight sparsity")

abs_W = W.abs()
thresholds = [1e-3, 1e-2, 5e-2, 1e-1]
print("What this means:")
print("- Fewer active basis dimensions per predicate means W is more sparse/compositional.")
print("- If every predicate activates every basis, increase LAMBDA_SPARSE or reduce RANK.")

for threshold in thresholds:
    active_counts = (abs_W > threshold).sum(dim=1)
    print({
        "threshold": threshold,
        "mean_active_basis": round(float(active_counts.float().mean()), 3),
        "median_active_basis": round(float(active_counts.float().median()), 3),
        "min_active_basis": int(active_counts.min()),
        "max_active_basis": int(active_counts.max()),
    })

l1_per_predicate = abs_W.sum(dim=1)
max_weight_per_predicate = abs_W.max(dim=1).values
print("\nW magnitude stats:")
print("mean L1 per predicate:", float(l1_per_predicate.mean()))
print("median L1 per predicate:", float(l1_per_predicate.median()))
print("mean max |weight| per predicate:", float(max_weight_per_predicate.mean()))

plt.figure(figsize=(6, 4))
plt.hist(l1_per_predicate.numpy(), bins=30)
plt.title("L1 norm of W per predicate")
plt.xlabel("||w_i||_1")
plt.ylabel("# predicates")
plt.show()

active_counts = (abs_W > 1e-2).sum(dim=1)
plt.figure(figsize=(6, 4))
plt.hist(active_counts.numpy(), bins=range(0, rank + 2))
plt.title("Active basis count per predicate, threshold=1e-2")
plt.xlabel("# active basis dimensions")
plt.ylabel("# predicates")
plt.show()


## 7. Diagnostics B: does the decomposition reveal useful predicate semantics?

This section checks whether the learned representation is semantically meaningful, not just numerically reconstructive.

We compare three spaces:

- `F`: original CLIP description features.
- `F_hat = W @ B.T`: low-rank reconstructed features.
- `W`: sparse predicate weights over shared basis features.

Useful signs:

- Similar predicates become neighbors in `W` or `F_hat`.
- Each basis dimension has interpretable high-weight predicates.
- Fallback descriptions are easy to identify, because they may be less semantically rich than future LLM descriptions.


In [ ]:
def cosine_similarity_matrix(x: torch.Tensor) -> torch.Tensor:
    x = F.normalize(x.float(), dim=-1)
    return x @ x.t()


def topk_neighbors(similarity: torch.Tensor, row_id: int, k: int = 5):
    values, indices = torch.topk(similarity[row_id], k=min(k + 1, similarity.shape[0]))
    neighbors = []
    for value, index in zip(values.tolist(), indices.tolist()):
        if index == row_id:
            continue
        neighbors.append((index, value))
        if len(neighbors) == k:
            break
    return neighbors


sim_F = cosine_similarity_matrix(features.cpu())
sim_reconstructed = cosine_similarity_matrix(reconstructed)
sim_W = cosine_similarity_matrix(W)

print("Diagnostic B1: Nearest-neighbor predicate comparison")
print("What this means:")
print("- F neighbors show CLIP's original predicate-description similarity.")
print("- F_hat neighbors show the denoised low-rank semantic space.")
print("- W neighbors show predicates that use similar shared basis combinations.")
print("- If W/F_hat neighbors are semantically cleaner than F, the decomposition is useful for analysis.")

query_predicates = ["on", "under", "riding", "holding", "wearing", "near", "behind", "in front of"]
available = {normalize_key(name): i for i, name in enumerate(predicate_names)}
query_rows = [available[normalize_key(name)] for name in query_predicates if normalize_key(name) in available]
if not query_rows:
    query_rows = list(range(min(8, len(predicate_names))))

for row_id in query_rows:
    print("\nQuery:", {"predicate_index": predicate_indices[row_id], "predicate": predicate_names[row_id]})
    for label, sim in [("Original F", sim_F), ("Reconstructed F_hat", sim_reconstructed), ("Weight W", sim_W)]:
        neighbors = topk_neighbors(sim, row_id, k=5)
        formatted = [
            {"predicate": predicate_names[idx], "score": round(score, 4)}
            for idx, score in neighbors
        ]
        print(label, formatted)


In [ ]:
print("Diagnostic B2: Basis interpretation from W")
print("What this means:")
print("- For each basis dimension k, predicates with the largest positive/negative W[:, k] reveal what that basis may encode.")
print("- If the top predicates share a recognizable relation type, that basis is interpretable.")

TOPK_PER_BASIS = 8
for basis_id in range(rank):
    weights = W[:, basis_id]
    top_pos = torch.topk(weights, k=min(TOPK_PER_BASIS, len(predicate_names))).indices.tolist()
    top_neg = torch.topk(-weights, k=min(TOPK_PER_BASIS, len(predicate_names))).indices.tolist()
    print(f"\nBasis {basis_id}")
    print("  Top positive predicates:")
    for row_id in top_pos:
        print("   ", {"predicate": predicate_names[row_id], "weight": round(float(weights[row_id]), 4)})
    print("  Top negative predicates:")
    for row_id in top_neg:
        print("   ", {"predicate": predicate_names[row_id], "weight": round(float(weights[row_id]), 4)})


In [ ]:
print("Diagnostic B3: Predicate-basis activation table")
print("What this means:")
print("- This shows which basis dimensions each predicate relies on most.")
print("- Sparse, meaningful top dimensions support the claim that predicates are combinations of shared basis features.")

TOPK_BASIS_PER_PREDICATE = min(5, rank)
rows_to_show = query_rows[:]
if len(rows_to_show) < min(10, len(predicate_names)):
    rows_to_show += [i for i in range(len(predicate_names)) if i not in rows_to_show][: min(10, len(predicate_names)) - len(rows_to_show)]

for row_id in rows_to_show:
    weights = W[row_id]
    top_dims = torch.topk(weights.abs(), k=TOPK_BASIS_PER_PREDICATE).indices.tolist()
    print("\nPredicate:", {"predicate_index": predicate_indices[row_id], "predicate": predicate_names[row_id]})
    print("Description:", descriptions[row_id])
    print("Used fallback:", bool(fallback_mask[row_id]))
    print("Top basis activations:")
    for basis_id in top_dims:
        print(" ", {"basis": basis_id, "weight": round(float(weights[basis_id]), 4)})


In [ ]:
print("Diagnostic B4: 2D visualization of predicate spaces")
print("What this means:")
print("- These PCA plots are only exploratory, but clusters can reveal whether low-rank W creates cleaner semantic grouping.")
print("- Labels are omitted by default to keep the plot readable; inspect nearest-neighbor tables for exact semantics.")


def pca_2d(x: torch.Tensor) -> torch.Tensor:
    x = x.float()
    x = x - x.mean(dim=0, keepdim=True)
    u, s, vh = torch.linalg.svd(x, full_matrices=False)
    return x @ vh[:2].t()

coords_F = pca_2d(features.cpu())
coords_reconstructed = pca_2d(reconstructed)
coords_W = pca_2d(W)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, coords, title in [
    (axes[0], coords_F, "Original CLIP F"),
    (axes[1], coords_reconstructed, "Reconstructed F_hat"),
    (axes[2], coords_W, "Weight space W"),
]:
    colors = ["tab:orange" if used else "tab:blue" for used in fallback_mask]
    ax.scatter(coords[:, 0].numpy(), coords[:, 1].numpy(), s=20, c=colors, alpha=0.75)
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

fig.suptitle("Predicate semantic spaces; orange = fallback prompt")
plt.tight_layout()
plt.show()


In [ ]:
print("Diagnostic B5: Fallback prompt audit")
print("What this means:")
print("- Fallback rows use only predicate-name prompts. Once you add LLM descriptions, these should decrease.")
print("- If fallback rows dominate, semantic analysis mainly reflects predicate names, not rich descriptions.")

fallback_examples = [
    (idx, name, desc)
    for idx, name, desc, used_fallback in zip(predicate_indices, predicate_names, descriptions, fallback_mask)
    if used_fallback
]

print(f"Fallback rows: {len(fallback_examples)} / {len(predicate_names)}")
print("First fallback examples:")
fallback_examples[:20]


## 8. Diagnostics C: per-predicate triplet statistics on train/test

This section prints **triplet-level dataset statistics** for each predicate using Visual Genome annotations.

For each predicate, it summarizes:

- how many triplets appear in **train** and **test**,
- the most frequent `(subject, predicate, object)` triplets in each split,
- how much train and test overlap for that predicate.

This helps answer questions like:

- Is a predicate dominated by only a few object pairs?
- Does test mostly reuse train triplets, or is it compositional/long-tail?
- Are some predicates semantically broad while others are very narrow?


In [ ]:
def load_vg_label_info(dict_file: str):
    with open(dict_file, "r", encoding="utf-8") as handle:
        payload = json.load(handle)

    if "idx_to_label" in payload:
        idx_to_label = payload["idx_to_label"]
        if isinstance(idx_to_label, dict):
            ind_to_classes = [name for _, name in sorted(((int(k), v) for k, v in idx_to_label.items()), key=lambda x: x[0])]
        else:
            ind_to_classes = list(idx_to_label)
    elif "label_to_idx" in payload:
        label_to_idx = payload["label_to_idx"]
        ind_to_classes = [name for name, _ in sorted(label_to_idx.items(), key=lambda x: int(x[1]))]
    else:
        raise ValueError("VG dict file must contain idx_to_label or label_to_idx.")

    if "idx_to_predicate" in payload:
        idx_to_predicate = payload["idx_to_predicate"]
        if isinstance(idx_to_predicate, dict):
            ind_to_predicates = [name for _, name in sorted(((int(k), v) for k, v in idx_to_predicate.items()), key=lambda x: x[0])]
        else:
            ind_to_predicates = list(idx_to_predicate)
    elif "predicate_to_idx" in payload:
        predicate_to_idx = payload["predicate_to_idx"]
        ind_to_predicates = [name for name, _ in sorted(predicate_to_idx.items(), key=lambda x: int(x[1]))]
    else:
        raise ValueError("VG dict file must contain idx_to_predicate or predicate_to_idx.")

    return ind_to_classes, ind_to_predicates


def build_vg_triplet_statistics(roidb_file: str, dict_file: str, num_val_im: int = 5000):
    if not os.path.exists(roidb_file):
        raise FileNotFoundError(
            f"VG roidb file not found: {roidb_file}. Put VG-SGG-with-attri.h5 there or update VG_ROIDB_FILE."
        )
    if not os.path.exists(dict_file):
        raise FileNotFoundError(
            f"VG dict file not found: {dict_file}. Put VG-SGG-dicts-with-attri.json there or update VG_DICT_FILE."
        )

    ind_to_classes, ind_to_predicates = load_vg_label_info(dict_file)
    split_to_triplets = {"train": {}, "test": {}}
    split_to_predicate_counts = {"train": {}, "test": {}}

    with h5py.File(roidb_file, "r") as roi_h5:
        data_split = roi_h5["split"][:]
        img_to_first_box = roi_h5["img_to_first_box"][:]
        img_to_last_box = roi_h5["img_to_last_box"][:]
        img_to_first_rel = roi_h5["img_to_first_rel"][:]
        img_to_last_rel = roi_h5["img_to_last_rel"][:]
        all_labels = roi_h5["labels"][:, 0]
        all_relations = roi_h5["relationships"][:]
        all_predicates = roi_h5["predicates"][:, 0]

        def image_indices_for_split(split_name: str):
            split_flag = 2 if split_name == "test" else 0
            split_mask = data_split == split_flag
            split_mask &= img_to_first_box >= 0
            split_mask &= img_to_first_rel >= 0
            image_index = torch.from_numpy((split_mask.nonzero())[0]).tolist()
            if split_name == "train":
                image_index = image_index[num_val_im:]
            elif split_name == "val":
                image_index = image_index[:num_val_im]
            return image_index

        for split_name in ["train", "test"]:
            image_index = image_indices_for_split(split_name)
            predicate_triplets = {}
            predicate_counts = {}

            for img_id in image_index:
                rel_start = int(img_to_first_rel[img_id])
                rel_end = int(img_to_last_rel[img_id])
                box_start = int(img_to_first_box[img_id])
                if rel_start < 0:
                    continue

                rel_pairs = all_relations[rel_start: rel_end + 1] - box_start
                rel_preds = all_predicates[rel_start: rel_end + 1]
                gt_classes = all_labels[box_start: int(img_to_last_box[img_id]) + 1]

                for (sub_idx, obj_idx), pred_idx in zip(rel_pairs, rel_preds):
                    pred_idx = int(pred_idx)
                    sub_name = ind_to_classes[int(gt_classes[sub_idx])]
                    pred_name = ind_to_predicates[pred_idx]
                    obj_name = ind_to_classes[int(gt_classes[obj_idx])]
                    triplet = (sub_name, pred_name, obj_name)

                    predicate_triplets.setdefault(pred_name, {})
                    predicate_triplets[pred_name][triplet] = predicate_triplets[pred_name].get(triplet, 0) + 1
                    predicate_counts[pred_name] = predicate_counts.get(pred_name, 0) + 1

            split_to_triplets[split_name] = predicate_triplets
            split_to_predicate_counts[split_name] = predicate_counts

    return {
        "ind_to_classes": ind_to_classes,
        "ind_to_predicates": ind_to_predicates,
        "split_to_triplets": split_to_triplets,
        "split_to_predicate_counts": split_to_predicate_counts,
    }


def summarize_predicate_triplets(vg_stats, predicate_name: str, topk: int = 10):
    train_triplets = vg_stats["split_to_triplets"]["train"].get(predicate_name, {})
    test_triplets = vg_stats["split_to_triplets"]["test"].get(predicate_name, {})
    train_count = vg_stats["split_to_predicate_counts"]["train"].get(predicate_name, 0)
    test_count = vg_stats["split_to_predicate_counts"]["test"].get(predicate_name, 0)
    train_unique = len(train_triplets)
    test_unique = len(test_triplets)
    overlap = set(train_triplets.keys()) & set(test_triplets.keys())

    return {
        "predicate": predicate_name,
        "train_total": train_count,
        "test_total": test_count,
        "train_unique_triplets": train_unique,
        "test_unique_triplets": test_unique,
        "shared_unique_triplets": len(overlap),
        "train_overlap_ratio": 0.0 if train_unique == 0 else len(overlap) / train_unique,
        "test_overlap_ratio": 0.0 if test_unique == 0 else len(overlap) / test_unique,
        "train_topk": sorted(train_triplets.items(), key=lambda x: x[1], reverse=True)[:topk],
        "test_topk": sorted(test_triplets.items(), key=lambda x: x[1], reverse=True)[:topk],
    }


In [ ]:
if ENABLE_VG_TRIPLET_STATS:
    print("Step 8: Build Visual Genome train/test triplet statistics.")
    print("- roidb file:", VG_ROIDB_FILE)
    print("- dict file:", VG_DICT_FILE)
    print("- top-k triplets per predicate:", TOPK_TRIPLETS_PER_PREDICATE)
    print("What this step does:")
    print("- Reads subject/object labels and relation annotations from VG.")
    print("- Builds counts for each (subject, predicate, object) triplet in train and test.")
    print("- Lets you inspect which triplets dominate each predicate and how much test overlaps train.")

    vg_triplet_stats = build_vg_triplet_statistics(VG_ROIDB_FILE, VG_DICT_FILE)
    print("Done building VG triplet statistics.")
    print("Predicates with train stats:", len(vg_triplet_stats["split_to_triplets"]["train"]))
    print("Predicates with test stats:", len(vg_triplet_stats["split_to_triplets"]["test"]))
else:
    vg_triplet_stats = None
    print("ENABLE_VG_TRIPLET_STATS=False, skip dataset triplet statistics.")


In [ ]:
if vg_triplet_stats is not None:
    print("Diagnostic C1: Print train/test triplet statistics for predicates in the current factorization set.")
    print("What this means:")
    print("- train_total/test_total: total relation instances for this predicate in each split.")
    print("- unique_triplets: number of distinct (subject, predicate, object) combinations.")
    print("- overlap_ratio: how much of the unique test/train triplet set appears in both splits.")

    for predicate_name in predicate_names:
        summary = summarize_predicate_triplets(
            vg_stats=vg_triplet_stats,
            predicate_name=predicate_name,
            topk=TOPK_TRIPLETS_PER_PREDICATE,
        )
        print("\n" + "=" * 100)
        print("Predicate:", summary["predicate"])
        print({
            "train_total": summary["train_total"],
            "test_total": summary["test_total"],
            "train_unique_triplets": summary["train_unique_triplets"],
            "test_unique_triplets": summary["test_unique_triplets"],
            "shared_unique_triplets": summary["shared_unique_triplets"],
            "train_overlap_ratio": round(summary["train_overlap_ratio"], 4),
            "test_overlap_ratio": round(summary["test_overlap_ratio"], 4),
        })
        print("Top train triplets:")
        for triplet, count in summary["train_topk"]:
            print("  ", {"triplet": triplet, "count": int(count)})
        print("Top test triplets:")
        for triplet, count in summary["test_topk"]:
            print("  ", {"triplet": triplet, "count": int(count)})
else:
    print("vg_triplet_stats is None, so per-predicate triplet statistics are unavailable.")


In [ ]:
if vg_triplet_stats is not None:
    print("Diagnostic C2: Compact ranking of predicates by train/test frequency and compositional overlap")

    rows = []
    for predicate_name in predicate_names:
        summary = summarize_predicate_triplets(
            vg_stats=vg_triplet_stats,
            predicate_name=predicate_name,
            topk=TOPK_TRIPLETS_PER_PREDICATE,
        )
        rows.append({
            "predicate": predicate_name,
            "train_total": summary["train_total"],
            "test_total": summary["test_total"],
            "train_unique_triplets": summary["train_unique_triplets"],
            "test_unique_triplets": summary["test_unique_triplets"],
            "test_overlap_ratio": summary["test_overlap_ratio"],
        })

    rows_by_train = sorted(rows, key=lambda x: x["train_total"], reverse=True)
    rows_by_test_overlap = sorted(rows, key=lambda x: x["test_overlap_ratio"])

    print("Top 15 predicates by train frequency:")
    for row in rows_by_train[:15]:
        print(row)

    print("\nTop 15 predicates with lowest test overlap ratio (more compositional / less seen in train):")
    for row in rows_by_test_overlap[:15]:
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
else:
    print("vg_triplet_stats is None, skip compact predicate ranking.")


## 9. Save `.pt` factors and `.json` metadata


In [ ]:
def metadata_path(output_path: str) -> str:
    root, _ = os.path.splitext(output_path)
    return root + ".json"


os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)

output_payload = {
    "features": features,
    "W": W,
    "B": B,
    "M": M,
    "reconstructed": reconstructed,
    "predicate_names": predicate_names,
    "predicate_indices": predicate_indices,
    "descriptions": descriptions,
    "rank": rank,
    "reconstruction_error": float(factor_payload["reconstruction_error"]),
    "final_total_loss": float(factor_payload["final_total_loss"]),
    "final_recon_loss": float(factor_payload["final_recon_loss"]),
    "final_sparse_loss": float(factor_payload["final_sparse_loss"]),
    "final_ortho_loss": float(factor_payload["final_ortho_loss"]),
    "loss_history": factor_payload["loss_history"],
    "factorization_method": "learnable_hola",
}

metadata = {
    "predicate_json": PREDICATE_JSON,
    "descriptions": DESCRIPTIONS,
    "clip_model": CLIP_MODEL,
    "device": DEVICE,
    "normalize": NORMALIZE_FEATURES,
    "requested_rank": RANK,
    "effective_rank": rank,
    "factorization_steps": FACTORIZATION_STEPS,
    "factorization_lr": FACTORIZATION_LR,
    "lambda_sparse": LAMBDA_SPARSE,
    "lambda_ortho": LAMBDA_ORTHO,
    "seed": SEED,
    "num_predicates": len(predicate_names),
    "feature_dim": int(features.shape[1]),
    "fallback_count": int(sum(fallback_mask)),
    "fallback_mask": fallback_mask,
    "predicate_names": predicate_names,
    "predicate_indices": predicate_indices,
    "description_texts": descriptions,
    "reconstruction_error": float(factor_payload["reconstruction_error"]),
    "final_total_loss": float(factor_payload["final_total_loss"]),
    "final_recon_loss": float(factor_payload["final_recon_loss"]),
    "final_sparse_loss": float(factor_payload["final_sparse_loss"]),
    "final_ortho_loss": float(factor_payload["final_ortho_loss"]),
    "loss_history": factor_payload["loss_history"],
    "factorization_method": "learnable_hola",
}

torch.save(output_payload, OUTPUT)
META_OUTPUT = metadata_path(OUTPUT)
with open(META_OUTPUT, "w", encoding="utf-8") as handle:
    json.dump(metadata, handle, ensure_ascii=False, indent=2)

print("Saved factors:", OUTPUT)
print("Saved metadata:", META_OUTPUT)
